# Session 9 - Parallel Algorithm Implementation with CUDA
## CT225H - Parallel Programming with CUDA
### Nguyễn Thanh Trọng - B2305615

**Bài toán 1:** Tính tích vô hướng (Dot Product) của hai Vector sử dụng Reduction  
**Bài toán 2:** Chuyển vị Ma trận (Matrix Transpose) sử dụng Shared Memory

In [1]:
# Kiểm tra GPU trên Colab
!nvidia-smi

Tue Mar  3 01:32:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bài toán 1: Tích vô hướng (Dot Product)

Cho hai vector $A$ và $B$ có cùng kích thước $N = 1{,}000{,}000$.  
Tính: $Result = \sum_{i=0}^{N-1} A_i \times B_i$

**Phương pháp:**
1. Mỗi thread tính tích $A_i \times B_i$ cho phần tử được gán
2. Sử dụng **Parallel Reduction** với Shared Memory để cộng dồn kết quả trong mỗi block
3. Kết quả từ các block được cộng dồn bằng `atomicAdd` trên global memory

In [2]:
%%writefile dot_product.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

#define N 1000000
#define BLOCK_SIZE 256

// ===================== CPU Version =====================
float dotProductCPU(float *a, float *b, int n) {
    float result = 0.0f;
    for (int i = 0; i < n; i++) {
        result += a[i] * b[i];
    }
    return result;
}

// ===================== GPU Kernel =====================
// Parallel Reduction: mỗi block tính tổng riêng phần bằng shared memory,
// sau đó dùng atomicAdd để cộng dồn kết quả vào biến global.
__global__ void dotProductKernel(float *a, float *b, float *result, int n) {
    __shared__ float cache[BLOCK_SIZE];

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Mỗi thread tính tích của cặp phần tử tương ứng
    float temp = 0.0f;
    while (idx < n) {
        temp += a[idx] * b[idx];
        idx += blockDim.x * gridDim.x; // Grid-stride loop
    }
    cache[tid] = temp;
    __syncthreads();

    // Parallel Reduction trong shared memory (tree-based)
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride) {
            cache[tid] += cache[tid + stride];
        }
        __syncthreads();
    }

    // Thread 0 của mỗi block cộng kết quả vào biến global
    if (tid == 0) {
        atomicAdd(result, cache[0]);
    }
}

int main() {
    float *h_a, *h_b;
    float h_result_cpu, h_result_gpu;
    float *d_a, *d_b, *d_result;

    size_t bytes = N * sizeof(float);

    // Cấp phát bộ nhớ host
    h_a = (float *)malloc(bytes);
    h_b = (float *)malloc(bytes);

    // Khởi tạo dữ liệu
    srand(42);
    for (int i = 0; i < N; i++) {
        h_a[i] = (float)(rand() % 100) / 100.0f;
        h_b[i] = (float)(rand() % 100) / 100.0f;
    }

    // ==================== CPU ====================
    clock_t cpu_start = clock();
    h_result_cpu = dotProductCPU(h_a, h_b, N);
    clock_t cpu_end = clock();
    double cpu_time = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;

    printf("=== Dot Product (N = %d) ===\n", N);
    printf("[CPU] Result: %.6f\n", h_result_cpu);
    printf("[CPU] Time:   %.4f ms\n", cpu_time);

    // ==================== GPU ====================
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_result, sizeof(float));

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);
    cudaMemset(d_result, 0, sizeof(float)); // Khởi tạo kết quả = 0

    int numBlocks = (N + BLOCK_SIZE - 1) / BLOCK_SIZE;

    // CUDA Events để đo thời gian chính xác
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    dotProductKernel<<<numBlocks, BLOCK_SIZE>>>(d_a, d_b, d_result, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time = 0.0f;
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(&h_result_gpu, d_result, sizeof(float), cudaMemcpyDeviceToHost);

    printf("\n[GPU] Result: %.6f\n", h_result_gpu);
    printf("[GPU] Time:   %.4f ms\n", gpu_time);
    printf("\nSpeedup: %.2fx\n", cpu_time / gpu_time);

    // Kiểm tra sai số
    float error = fabs(h_result_cpu - h_result_gpu);
    printf("Absolute error: %.6f\n", error);

    // Giải phóng bộ nhớ
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_result);
    free(h_a);
    free(h_b);

    return 0;
}

Writing dot_product.cu


In [3]:
# Biên dịch và chạy Dot Product
!nvcc -o dot_product dot_product.cu -arch=sm_75
!./dot_product

=== Dot Product (N = 1000000) ===
[CPU] Result: 244166.156250
[CPU] Time:   3.0210 ms

[GPU] Result: 244218.968750
[GPU] Time:   0.1928 ms

Speedup: 15.67x
Absolute error: 52.812500


## Bài toán 2: Chuyển vị Ma trận (Matrix Transpose)

Cho ma trận $A$ kích thước $N \times M$ (với $N = M = 1024$).  
Tạo ma trận chuyển vị $A^T$ sao cho: $A^T[j][i] = A[i][j]$

**Phương pháp:**
1. **Naive Transpose:** Mỗi thread đọc 1 phần tử từ $A[i][j]$ và ghi vào $A^T[j][i]$
2. **Shared Memory Transpose (Optimized):** Sử dụng shared memory tile để đọc dữ liệu coalesced, sau đó ghi ra output coalesced → tránh non-coalesced global memory access
3. So sánh hiệu năng giữa CPU, Naive GPU, và Optimized GPU

In [4]:
%%writefile matrix_transpose.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

#define ROWS 1024
#define COLS 1024
#define TILE_DIM 32
#define BLOCK_ROWS 8

// ===================== CPU Version =====================
void transposeCPU(float *in, float *out, int rows, int cols) {
    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) {
            out[j * rows + i] = in[i * cols + j];
        }
    }
}

// ===================== GPU Naive Kernel =====================
// Mỗi thread đọc A[i][j] và ghi vào AT[j][i]
// Đọc coalesced nhưng ghi non-coalesced
__global__ void transposeNaive(float *in, float *out, int rows, int cols) {
    int x = blockIdx.x * TILE_DIM + threadIdx.x;
    int y = blockIdx.y * TILE_DIM + threadIdx.y;

    for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS) {
        if (x < cols && (y + j) < rows) {
            out[x * rows + (y + j)] = in[(y + j) * cols + x];
        }
    }
}

// ===================== GPU Shared Memory Kernel =====================
// Sử dụng shared memory tile: đọc coalesced vào shared memory,
// sau đó ghi coalesced ra global memory → tối ưu bandwidth
__global__ void transposeSharedMem(float *in, float *out, int rows, int cols) {
    __shared__ float tile[TILE_DIM][TILE_DIM + 1]; // +1 để tránh bank conflict

    int x = blockIdx.x * TILE_DIM + threadIdx.x;
    int y = blockIdx.y * TILE_DIM + threadIdx.y;

    // Đọc dữ liệu vào shared memory (coalesced read)
    for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS) {
        if (x < cols && (y + j) < rows) {
            tile[threadIdx.y + j][threadIdx.x] = in[(y + j) * cols + x];
        }
    }
    __syncthreads();

    // Tính toán vị trí output (đổi block index)
    x = blockIdx.y * TILE_DIM + threadIdx.x;
    y = blockIdx.x * TILE_DIM + threadIdx.y;

    // Ghi dữ liệu từ shared memory ra global memory (coalesced write)
    for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS) {
        if (x < rows && (y + j) < cols) {
            out[(y + j) * rows + x] = tile[threadIdx.x][threadIdx.y + j];
        }
    }
}

// ===================== Verification =====================
int verify(float *ref, float *result, int n) {
    for (int i = 0; i < n; i++) {
        if (fabs(ref[i] - result[i]) > 1e-5) {
            printf("Mismatch at index %d: ref=%.6f, result=%.6f\n", i, ref[i], result[i]);
            return 0;
        }
    }
    return 1;
}

int main() {
    int rows = ROWS, cols = COLS;
    size_t bytes = rows * cols * sizeof(float);

    // Cấp phát bộ nhớ host
    float *h_input  = (float *)malloc(bytes);
    float *h_cpu    = (float *)malloc(bytes);
    float *h_naive  = (float *)malloc(bytes);
    float *h_shared = (float *)malloc(bytes);

    // Khởi tạo dữ liệu
    srand(42);
    for (int i = 0; i < rows * cols; i++) {
        h_input[i] = (float)(rand() % 1000) / 100.0f;
    }

    // ==================== CPU ====================
    clock_t cpu_start = clock();
    transposeCPU(h_input, h_cpu, rows, cols);
    clock_t cpu_end = clock();
    double cpu_time = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;

    printf("=== Matrix Transpose (%d x %d) ===\n", rows, cols);
    printf("[CPU]          Time: %.4f ms\n", cpu_time);

    // Cấp phát bộ nhớ device
    float *d_input, *d_output;
    cudaMalloc(&d_input, bytes);
    cudaMalloc(&d_output, bytes);
    cudaMemcpy(d_input, h_input, bytes, cudaMemcpyHostToDevice);

    dim3 block(TILE_DIM, BLOCK_ROWS);
    dim3 grid((cols + TILE_DIM - 1) / TILE_DIM, (rows + TILE_DIM - 1) / TILE_DIM);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float gpu_time;

    // ==================== GPU Naive ====================
    cudaMemset(d_output, 0, bytes);
    cudaEventRecord(start);
    transposeNaive<<<grid, block>>>(d_input, d_output, rows, cols);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(h_naive, d_output, bytes, cudaMemcpyDeviceToHost);
    printf("[GPU Naive]    Time: %.4f ms  |  Correct: %s\n",
           gpu_time, verify(h_cpu, h_naive, rows * cols) ? "Yes" : "No");
    float naive_time = gpu_time;

    // ==================== GPU Shared Memory ====================
    cudaMemset(d_output, 0, bytes);
    cudaEventRecord(start);
    transposeSharedMem<<<grid, block>>>(d_input, d_output, rows, cols);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(h_shared, d_output, bytes, cudaMemcpyDeviceToHost);
    printf("[GPU Shared]   Time: %.4f ms  |  Correct: %s\n",
           gpu_time, verify(h_cpu, h_shared, rows * cols) ? "Yes" : "No");
    float shared_time = gpu_time;

    // ==================== Summary ====================
    printf("\n--- Performance Summary ---\n");
    printf("CPU Time:              %.4f ms\n", cpu_time);
    printf("GPU Naive Time:        %.4f ms  (Speedup: %.2fx vs CPU)\n", naive_time, cpu_time / naive_time);
    printf("GPU Shared Mem Time:   %.4f ms  (Speedup: %.2fx vs CPU)\n", shared_time, cpu_time / shared_time);
    printf("Shared vs Naive:       %.2fx faster\n", naive_time / shared_time);

    // In một phần ma trận để kiểm tra
    printf("\n--- Verification (top-left 4x4) ---\n");
    printf("Original:\n");
    for (int i = 0; i < 4; i++) {
        for (int j = 0; j < 4; j++)
            printf("%7.2f ", h_input[i * cols + j]);
        printf("\n");
    }
    printf("Transposed (GPU Shared):\n");
    for (int i = 0; i < 4; i++) {
        for (int j = 0; j < 4; j++)
            printf("%7.2f ", h_shared[i * rows + j]);
        printf("\n");
    }

    // Giải phóng bộ nhớ
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_input);
    cudaFree(d_output);
    free(h_input);
    free(h_cpu);
    free(h_naive);
    free(h_shared);

    return 0;
}

Writing matrix_transpose.cu


In [5]:
# Biên dịch và chạy Matrix Transpose
!nvcc -o matrix_transpose matrix_transpose.cu -arch=sm_75
!./matrix_transpose

=== Matrix Transpose (1024 x 1024) ===
[CPU]          Time: 19.7390 ms
[GPU Naive]    Time: 0.2270 ms  |  Correct: Yes
[GPU Shared]   Time: 0.0619 ms  |  Correct: Yes

--- Performance Summary ---
CPU Time:              19.7390 ms
GPU Naive Time:        0.2270 ms  (Speedup: 86.94x vs CPU)
GPU Shared Mem Time:   0.0619 ms  (Speedup: 318.95x vs CPU)
Shared vs Naive:       3.67x faster

--- Verification (top-left 4x4) ---
Original:
   1.66    7.40    8.81    2.41 
   3.30    3.39    5.14    8.54 
   8.64    7.95    0.92    8.08 
   7.05    6.73    4.58    0.41 
Transposed (GPU Shared):
   1.66    3.30    8.64    7.05 
   7.40    3.39    7.95    6.73 
   8.81    5.14    0.92    4.58 
   2.41    8.54    8.08    0.41 
